# Access SOFAR data from HPSS archive, convert BUFR files to netCDF, and compute matchups between GFS and SOFAR data

In [2]:
import subprocess
import os
import glob
import datetime
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import netCDF4 as nc
import xarray as xr
import ncepbufr
from dask.distributed import Client
import icechunk
from icechunk.xarray import to_icechunk
from IPython.display import display

%load_ext autoreload
%autoreload 2

import ww3tools.sofar as sf

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Access the SOFAR data from the NOAA HPSS system for given dates


In [ ]:
# Define start and end dates 
start_date = "2026-01-01"
end_date = "2026-01-31"

# Define the working data directory to save the data
working_dir = "/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/bufr_data/"

# Define slurm parameters
slurm_params = {
            "ntasks": 1,
            "queue": "windfall",
            "partition": "u1-service",
            "walltime": "08:00:00",
            "account": "marine-cpu",
            "job_name": "hpss_sofar"
            }

# Access the data
sf.get_sofar_hpss(start_date, end_date, slurm_params, working_dir)

## Convert the BUFR Format files to netCDF4 file formats

In [ ]:
# Define paths on Ursa
bufr_dir = '/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/bufr_data'
output_dir = '/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/nc_data'

start_date = "2026-01-01"
end_date = "2026-03-10"

# Run batch conversion
nc_files = sf.convert_sofar_bufr_to_nc(
    bufr_dir=bufr_dir,
    output_dir=output_dir,
    start_date=start_date,
    end_date=end_date,
    pattern="sofar_b001_xx121_*"
)

print(f"Generated {len(nc_files)} NetCDF files.")

In [4]:
# Open an example netCDF file to view contents
ds = xr.open_dataset('/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/nc_data/sofar_b001_xx121_20260101.nc')
display(ds)
ds.close()

<xarray.Dataset> Size: 688kB
Dimensions:                                                    (obs: 5730)
Coordinates:
  * obs                                                        (obs) int64 46kB ...
    time                                                       (obs) datetime64[ns] 46kB ...
Data variables:
    platform_id                                                (obs) <U8 183kB ...
    latitude                                                   (obs) float64 46kB ...
    longitude                                                  (obs) float64 46kB ...
    sea_surface_wave_significant_height                        (obs) float64 46kB ...
    wind_from_direction                                        (obs) float64 46kB ...
    wind_speed                                                 (obs) float64 46kB ...
    sea_surface_wave_directional_spread                        (obs) float64 46kB ...
    sea_surface_wave_from_direction_at_variance_spectral_peak  (obs) float64 46kB ...
    sea_surface_wave_mean_period                               (obs) float64 46kB ...
    sea_surface_wave_period_at_variance_spectral_peak          (obs) float64 46kB ...
Attributes:
    Conventions:  CF-1.9
    featureType:  trajectory
    title:        Surface Marine Observation Data Converted from NCEP BUFR
    source:       NCEP BUFR observation file: sofar_b001_xx121_20260101
    history:      2026-10-06T19:18:39Z: Converted to CF-1.9 NetCDF via ww3-to...

## Combine Sofar data and set up for Interpolations

In [ ]:
# Get SOFAR netCDF file names
sofar_files = sorted(glob.glob("/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/nc_data/sofar_*.nc"))
chunk_size = {"time": 500}

ds_sofar = xr.open_mfdataset(
   sofar_files,
   combine="nested",
   concat_dim= "obs",
   parallel=True,
   chunks={"obs": 500},
   engine="netcdf4",
)

# Ensure coordinates are properly sorted by time
ds_sofar = ds_sofar.sortby("time")

# Display the combined SOFAR dataset
ds_sofar

In [ ]:
# 4. Check original longitude range
lon_min = ds_sofar.longitude.min().compute().item()
lon_max = ds_sofar.longitude.max().compute().item()
print(f"Original longitude range: {lon_min:.2f} to {lon_max:.2f}")

# 5. Convert longitude from [-180, 180] to [0, 360] format
ds_sofar["longitude"] = (ds_sofar["longitude"] + 360) % 360

# 6. Verify the shift worked
new_lon_min = ds_sofar.longitude.min().compute().item()
new_lon_max = ds_sofar.longitude.max().compute().item()
print(f"New longitude range: {new_lon_min:.2f} to {new_lon_max:.2f}")

In [ ]:
# Load a GFS forecast file to check all variables
gfs_path = '/scratch3/NCEPDEV/climate/Jessica.Meixner/Data/retrov17_01/gfs.20260101/00/products/wave/gridded/global.0p25/gfs.t00z.global.0p25.f000.grib2'

# Open the grib file into an xarray dataset
gfs_ds = xr.open_dataset(gfs_path, engine="cfgrib", backend_kwargs={'indexpath':''})

# Display the gfs data
ds_gfs

Variable matchups: 
1. significant wave height: sofar (sea_surface_wave_significant_height) and  gfs (swh)
2. wind speed: sofar (wind_speed) and gfs (ws)
3. wind direction: sofar (wind_from_direction) and gfs (wdir)
4. mean wave period: sofar (sea_surface_wave_mean_period) and gfs (mp1)
5. mean wave direction: sofar (sea_surface_wave_from_direction_at_variance_spectral_peak) and gfs (wvdir)

# Test code for building matchup file

In [ ]:
# Pick the first forecast cycle directory from your folder structure
test_cycle_dir = "/scratch3/NCEPDEV/climate/Jessica.Meixner/Data/retrov17_01/gfs.20260101/00/products/wave/gridded/global.0p25"

# Extract cycle initialization time (e.g., 2026-01-01 00:00:00)
parts = Path(test_cycle_dir).parts
gfs_date = [p for p in parts if p.startswith("gfs.")][0].split(".")[1]
gfs_hour = parts[parts.index(f"gfs.{gfs_date}") + 1]
init_time = pd.to_datetime(f"{gfs_date} {gfs_hour}:00:00")
print(f"Processing cycle: {init_time}")

# Get all forecast hour files for this cycle
f_files = sorted(glob.glob(f"{test_cycle_dir}/gfs.t{gfs_hour}z.global.0p25.f*.grib2"))

# Lazy load the entire forecast cycle
ds_gfs = xr.open_mfdataset(
    f_files,
    engine="cfgrib",
    combine="nested",
    concat_dim="step",
    coords="minimal",
    compat="override",
    chunks={},  
    parallel=True,
    backend_kwargs={"filter_by_keys": {"typeOfLevel": "surface"}, "indexpath": ""}
)

# Create continuous 'valid_time' coordinate and swap dimensions
# Drop the singleton 'time' dimension if cfgrib kept it (we only need 'step')
if "time" in ds_gfs.dims:
    ds_gfs = ds_gfs.squeeze("time")

# Forcefully overwrite valid_time to be exactly 1D along 'step'
ds_gfs = ds_gfs.assign_coords(
    valid_time=("step", (np.datetime64(init_time) + ds_gfs["step"]).data)
)

# Now swap dimensions safely
ds_gfs = ds_gfs.swap_dims({"step": "valid_time"}).sortby("valid_time")

# Subset GFS to only the variables we need
gfs_vars = ["swh", "ws", "wdir", "mp1", "wvdir"]
ds_gfs = ds_gfs[gfs_vars]

# Subset SOFAR data to match this 16-day forecast window
start_time = ds_gfs.valid_time.min().values
end_time = ds_gfs.valid_time.max().values
ds_sofar_sub = ds_sofar.sel(time=slice(start_time, end_time))

if len(ds_sofar_sub.time) == 0:
    print("No SOFAR data found for this specific 16-day window. Try testing a different cycle!")
else:
    # Consolidate chunks to prevent Dask fragmentation during interpolation
    ds_gfs = ds_gfs.chunk({"valid_time": -1})
    
    # Dynamically find the primary dimension of the SOFAR dataset and chunk it
    sofar_dim = list(ds_sofar_sub.sizes.keys())[0]  # This will grab 'obs' or 'time'
    ds_sofar_sub = ds_sofar_sub.chunk({sofar_dim: -1})
    
    print(f"Found {len(ds_sofar_sub[sofar_dim])} buoy observations. Interpolating...")

    # Perform Vectorized Interpolation!
    ds_model_interp = ds_gfs.interp(
        valid_time=ds_sofar_sub.time,
        latitude=ds_sofar_sub.latitude,
        longitude=ds_sofar_sub.longitude,
        method="linear"
    )

    # Build the final unified dataset matching your target NetCDF output
    fcst_hr_vals = (ds_sofar_sub.time.values - np.datetime64(init_time)) / np.timedelta64(1, 'h')
    
    ds_combined = xr.Dataset(
        data_vars={
            "latitude": ("time", ds_sofar_sub["latitude"].data),
            "longitude": ("time", (ds_sofar_sub["longitude"] % 360).data), 
            "fcst_hr": ("time", fcst_hr_vals.data),
            
            # Matched Variables
            "model_hs": ("time", ds_model_interp["swh"].data),
            "obs_hs": ("time", ds_sofar_sub["sea_surface_wave_significant_height"].data),
            
            "model_wnd": ("time", ds_model_interp["ws"].data),
            "obs_wnd": ("time", ds_sofar_sub["wind_speed"].data),
            
            "model_wdir": ("time", ds_model_interp["wdir"].data),
            "obs_wdir": ("time", ds_sofar_sub["wind_from_direction"].data),
            
            "model_mwp": ("time", ds_model_interp["mp1"].data),
            "obs_mwp": ("time", ds_sofar_sub["sea_surface_wave_mean_period"].data),
            
            "model_mwdir": ("time", ds_model_interp["wvdir"].data),
            "obs_mwdir": ("time", ds_sofar_sub["sea_surface_wave_from_direction_at_variance_spectral_peak"].data),
        },
        coords={"time": ds_sofar_sub.time.data},
        attrs={
            "obs_platform": "SOFAR Spotter Buoy",
            "model_name": "GFS_WAVEv17 retro",
            "initialization_time": str(init_time)
        }
    )
    
    # Compute the final dataset into memory so we can view it
    ds_combined = ds_combined.compute()
    print("\nSuccess! Final Combined Dataset:")
    
    print(ds_combined)

This code is then refactored into a function called process_single_cycle which is tested in the following cell to make sure the output is expected.

## Test matchup function

In [ ]:
test_cycle_dir = "/scratch3/NCEPDEV/climate/Jessica.Meixner/Data/retrov17_01/gfs.20260101/00/products/wave/gridded/global.0p25"
init_time = pd.to_datetime("2026-01-01 00:00:00")
print(f"\nTesting worker function on cycle: {init_time}...")

# 4. Call the delayed function
delayed_task = sf.process_single_cycle(test_cycle_dir, init_time, ds_sofar)

# 5. Execute the Dask task
test_result = delayed_task.compute()

# 6. Verify output
if test_result is not None:
    print("\nSuccess! Worker returned the matched Dataset:")
    print(test_result)
else:
    print("\nFunction returned None (Missing GRIB files or no SOFAR data in this window).")

## Expand the interpolations to the full month of January

In [ ]:
# Start Dask cluster (if not already running)
try:
    client = Client()
except ValueError:
    pass

# Define inputs
start_date = "2026-01-01"
end_date = "2026-01-31"
sofar_dir = "/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/nc_data"
model_dir = "/scratch3/NCEPDEV/climate/Jessica.Meixner/Data/retrov17_01"
icechunk_store = "/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/sofar_gfs_matchup_store"
netcdf_path = "/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/sofar_gfs_matchup_jan2026.nc"
model_name = 'GFSv17 retro'

# Run the entire parallel pipeline
ds_final = sf.process_matchup_pipeline(start_date, end_date, sofar_dir, model_dir, model_name)

# Save the final dataset to Icechunk
print("Writing final dataset to Icechunk...")

ds_final = ds_final.chunk({"obs": 50000})
storage = icechunk.local_filesystem_storage(icechunk_store)
repo = icechunk.Repository.open_or_create(storage=storage)
session = repo.writable_session(branch="main")
to_icechunk(ds_final, session, mode="w")

session.commit(f"Appended GFS-SOFAR data {start_date} to {end_date}")
print("Icechunk write complete!")


print(f"\nWriting dataset to NetCDF at {netcdf_path}...")    
# Save to NetCDF. Xarray automatically coordinates the Dask workers to write this file.
ds_final.to_netcdf(netcdf_path)
print("NetCDF write complete!")
print("Done! Pipeline complete.")

--- Starting Pipeline ---
Target Date Range: 2026-01-01 to 2026-01-31
Loading SOFAR Buoy data...
Found 68 SOFAR files.
Concatenating SOFAR observations (preserving all duplicates)...
Writing SOFAR reference to disk at ./temp_sofar_reference.zarr...
Scanning GFS forecast cycle directories...
Found 2699 total cycles on disk.
Filtered to 121 forecast cycles within the date range.
Computing in parallel across Dask workers...


/scratch3/NCEPDEV/marine/Edwin.Rainville/envs/sofar/lib/python3.14/site-packages/distributed/client.py:3429: UserWarning: Sending large graph of size 10.17 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/scratch3/NCEPDEV/marine/Edwin.Rainville/envs/sofar/lib/python3.14/site-packages/distributed/client.py:3429: UserWarning: Sending large graph of size 10.25 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/scratch3/NCEPDEV/marine/Edwin.Rainville/envs/sofar/lib/python3.14/site-packages/distributed/client.py:3429:

In [6]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import icechunk
from icechunk.xarray import to_icechunk

# 1. Configure the Dask SLURM Cluster
cluster = SLURMCluster(
    queue='u1-compute',               # SLURM partition (e.g., 'batch', 'hera', 'orion')
    project='marine-cpu',             # **MUST UPDATE**: Your HPC allocation account
    cores=40,                         # Number of CPU cores per node
    memory='16GB',                    # Total RAM requested per node
    walltime='02:00:00',              # Max run time for the Dask workers
    processes=4,                      # Splits the 40 cores into 4 Python workers per node
    local_directory='/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data', # Fast scratch storage
    job_extra_directives=['--qos=batch']
)

2026-10-07 20:45:47,601 - distributed.scheduler - CRITICAL - Closed comm <BatchedSend: closed> while trying to write [{'op': 'key-in-memory', 'key': 'finalize-hlgfinalizecompute-8e68299147024b208a16483cf9d0159d', 'type': b'\x80\x05\x95\x15\x00\x00\x00\x00\x00\x00\x00\x8c\x05numpy\x94\x8c\x07ndarray\x94\x93\x94.'}]
Traceback (most recent call last):
  File "/scratch3/NCEPDEV/marine/Edwin.Rainville/envs/sofar/lib/python3.14/site-packages/distributed/scheduler.py", line 6307, in send_all
    c.send(*msgs)
    ~~~~~~^^^^^^^
  File "/scratch3/NCEPDEV/marine/Edwin.Rainville/envs/sofar/lib/python3.14/site-packages/distributed/batched.py", line 156, in send
    raise CommClosedError(f"Comm {self.comm!r} already closed.")
distributed.comm.core.CommClosedError: Comm <TCP (closed) Scheduler->Client local=tcp://10.212.1.11:41069 remote=tcp://10.212.2.27:41266> already closed.
2026-10-07 20:45:47,764 - distributed.scheduler - CRITICAL - Closed comm <BatchedSend: closed> while trying to write [{'op'

In [7]:
# 2. Scale the cluster 
# jobs=4 submits 4 SLURM batch jobs, giving you 160 total cores (4 jobs * 40 cores)
cluster.scale(jobs=4)
client = Client(cluster)

print(f"SLURM Dask Dashboard: {client.dashboard_link}")
print("Waiting for workers to spin up...")

# 3. Define inputs
start_date = "2026-01-01"
end_date = "2026-01-31"
sofar_dir = "/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/nc_data"
model_dir = "/scratch3/NCEPDEV/climate/Jessica.Meixner/Data/retrov17_01"
icechunk_store = "/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/sofar_gfs_matchup_store"
netcdf_path = "/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/sofar_gfs_matchup_jan2026.nc"
model_name = 'GFSv17 retro'

# 4. Run the entire parallel pipeline
# Dask will automatically distribute the tasks to your 160 SLURM cores
ds_final = sf.process_matchup_pipeline(start_date, end_date, sofar_dir, model_dir, model_name)

# 5. Save the final dataset to Icechunk
print("\nWriting final dataset to Icechunk...")
ds_final = ds_final.chunk({"obs": 50000})
storage = icechunk.local_filesystem_storage(icechunk_store)
repo = icechunk.Repository.open_or_create(storage=storage)
session = repo.writable_session(branch="main")

to_icechunk(ds_final, session, mode="w")
session.commit(f"Appended GFS-SOFAR data {start_date} to {end_date}")
print("Icechunk write complete!")

# 6. Save to NetCDF
print(f"\nWriting dataset to NetCDF at {netcdf_path}...")    
ds_final.to_netcdf(netcdf_path)
print("NetCDF write complete!")
print("Done! Pipeline complete.")

SLURM Dask Dashboard: http://10.212.1.11:46015/status
Waiting for workers to spin up...
--- Starting Pipeline ---
Target Date Range: 2026-01-01 to 2026-01-31
Loading SOFAR Buoy data...
Found 68 SOFAR files.
Concatenating SOFAR observations (preserving all duplicates)...
Writing SOFAR reference to disk at ./temp_sofar_reference.zarr...
Scanning GFS forecast cycle directories...
Found 2699 total cycles on disk.
Filtered to 121 forecast cycles within the date range.
Computing in parallel across Dask workers...


KilledWorker: Attempted to run task 'process_single_cycle-e35f4a92-01d4-447e-89be-48df64b3b5fa' on 4 different workers, but all those workers died while running it. The last worker that attempt to run the task was tcp://10.212.2.27:35913. Inspecting worker logs is often a good next step to diagnose what went wrong. For more information see https://distributed.dask.org/en/stable/killed.html.

In [ ]:
ds_final = ds_final.chunk({"obs": 50000})
storage = icechunk.local_filesystem_storage(icechunk_store)
repo = icechunk.Repository.open_or_create(storage=storage)
session = repo.writable_session(branch="main")
to_icechunk(ds_final, session, mode="w")

session.commit(f"Appended GFS-SOFAR data {start_date} to {end_date}")
print("Icechunk write complete!")


print(f"\nWriting dataset to NetCDF at {netcdf_path}...")    
# Save to NetCDF. Xarray automatically coordinates the Dask workers to write this file.
ds_final.to_netcdf(netcdf_path)
print("NetCDF write complete!")
print("Done! Pipeline complete.")

In [ ]:
ds_final

In [ ]:
# Chunk for Zarr storage
# Define inputs
start_date = "2026-01-01"
end_date = "2026-01-03"
sofar_dir = "/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/nc_data"
model_dir = "/scratch3/NCEPDEV/climate/Jessica.Meixner/Data/retrov17_01"
icechunk_store = "/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/sofar_gfs_matchup_store"
model_name = 'GFSv17 retro'

ds_final = ds_final.chunk({"obs": 50000})
storage = icechunk.local_filesystem_storage(icechunk_store)
repo = icechunk.Repository.open_or_create(storage=storage)
session = repo.writable_session(branch="main")
to_icechunk(ds_final, session)

session.commit(f"Appended GFS-SOFAR data {start_date} to {end_date}")
print("Done! Pipeline complete.")

In [ ]:
if ds_final is not None:
    netcdf_path = "/scratch3/NCEPDEV/marine/Edwin.Rainville/ww3-tools-dev/data/sofar_gfs_matchup_jan2026.nc"
    print(f"\nWriting dataset to NetCDF at {netcdf_path}...")
    
    # Save to NetCDF. Xarray automatically coordinates the Dask workers to write this file.
    ds_final.to_netcdf(netcdf_path)
    print("NetCDF write complete!")